# Local CPU inference smoke
This notebook reads only the five allowlisted inference derivatives and their manifest. It requires PyArrow 21.0.0 and performs no model, GPU, API or network work.
The management command runs these same code cells in two fresh isolated Python processes. Set `INFERENCE_ROOT` to the inference directory and `SMOKE_IDS` to the two opaque IDs from the environment receipt before manual execution. Selection of train IDs happens outside this notebook. Saved cell outputs and execution counts remain empty.
Full schema validation is recorded separately in the management environment receipt. The smoke validates manifest hashes, counts and incident joins and hashes selected records.


In [ ]:
from pathlib import Path
import hashlib
import json
import pyarrow.parquet as pq

# The management caller supplies these two opaque IDs after its private train
# selection. For manual execution, copy only IDs from the environment receipt.
if "INFERENCE_ROOT" not in globals() or "SMOKE_IDS" not in globals():
    raise ValueError("SMOKE_PARAMETERS_REQUIRED")
root = Path(INFERENCE_ROOT).resolve()
selected_ids = sorted(SMOKE_IDS)
if len(selected_ids) != 2 or len(set(selected_ids)) != 2:
    raise ValueError("SMOKE_EXPECTS_TWO_DISTINCT_IDS")


def safe_file(name):
    if Path(name).name != name:
        raise ValueError("SMOKE_MANIFEST_PATH")
    path = (root / name).resolve(strict=True)
    if path.parent != root or not path.is_file():
        raise ValueError("SMOKE_MANIFEST_PATH")
    return path


def reject_nonfinite(value):
    raise ValueError("SMOKE_NONFINITE_JSON")


def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"),
                      ensure_ascii=False, allow_nan=False).encode("utf-8")


manifest_path = safe_file("input-manifest.json")
manifest_bytes = manifest_path.read_bytes()
manifest = json.loads(manifest_bytes, parse_constant=reject_nonfinite)
expected_files = {"incident-index.parquet", "observations.jsonl",
                  "logs-evidence.jsonl", "metric-summaries.jsonl", "trace-evidence.jsonl"}
entries = manifest["files"]
if len(entries) != 5 or {entry["path"] for entry in entries} != expected_files:
    raise ValueError("SMOKE_MANIFEST_ALLOWLIST")
index = pq.read_table(safe_file("incident-index.parquet"))
index_rows = index.to_pylist()
all_ids = [row["incident_id"] for row in index_rows]
if len(all_ids) != 90 or len(set(all_ids)) != 90 or manifest["incident_count"] != 90:
    raise ValueError("SMOKE_INDEX_IDS")
if not set(selected_ids).issubset(set(all_ids)):
    raise ValueError("SMOKE_UNKNOWN_ID")
selected_records = {}
file_hashes = {}
row_counts = {}
selected_counts = {}
for entry in sorted(entries, key=lambda item: item["path"]):
    name = entry["path"]
    content = safe_file(name).read_bytes()
    digest = hashlib.sha256(content).hexdigest()
    if digest != entry["sha256"] or len(content) != entry["bytes"]:
        raise ValueError("SMOKE_FILE_HASH")
    if name.endswith(".parquet"):
        rows = index_rows
    else:
        rows = [json.loads(line, parse_constant=reject_nonfinite)
                for line in content.decode("utf-8").splitlines() if line.strip()]
    if len(rows) != entry["rows"]:
        raise ValueError("SMOKE_ROW_COUNT")
    if any(row.get("incident_id") not in all_ids for row in rows):
        raise ValueError("SMOKE_FOREIGN_INCIDENT")
    if name == "observations.jsonl":
        observation_ids = [row["incident_id"] for row in rows]
        if len(observation_ids) != 90 or set(observation_ids) != set(all_ids):
            raise ValueError("SMOKE_OBSERVATION_IDS")
    selected = [row for row in rows if row["incident_id"] in selected_ids]
    if {row["incident_id"] for row in selected} != set(selected_ids):
        raise ValueError("SMOKE_MISSING_EVIDENCE")
    selected_records[name] = sorted(selected, key=canonical)
    file_hashes[name] = digest
    row_counts[name] = len(rows)
    selected_counts[name] = len(selected)
smoke_result = {
    "status": "pass", "incident_ids": selected_ids,
    "manifest_sha256": hashlib.sha256(manifest_bytes).hexdigest(),
    "data_content_hash": manifest["content_hash"],
    "selected_content_hash": hashlib.sha256(canonical(selected_records)).hexdigest(),
    "file_sha256": file_hashes, "row_counts": row_counts,
    "selected_record_counts": selected_counts,
    "checks": ["manifest_file_allowlist", "canonical_paths", "file_hashes_and_sizes",
               "declared_row_counts", "90_unique_incident_ids", "observation_index_join",
               "two_incidents_present_in_all_derivatives", "finite_json"]
}
# Deliberately emit only aggregate counts, hashes and opaque IDs.
print(json.dumps(smoke_result, sort_keys=True))
